# 青木まりこ現象｜分析② ATUSの場所別活動時間

**研究目的**：書店以外の施設における活動時間を比較し、「書店の滞在時間が特別に長いから便意が起きるだけ」という説明を検討するための基礎資料を作る。

- データ：米国労働統計局（BLS）**American Time Use Survey (ATUS)** 公開個票（**2023・2024・2025年**）
- 原データ：[ATUS 公開データ一覧](https://www.bls.gov/tus/data.htm)
- 解析項目：`TUCASEID`（調査参加者）、`TEWHERE`（場所）、`TUACTDUR24`（活動時間［分］）、`TUFINLWGT`（調査ウェイト）
- 比較：図書館（10）、その他店舗・モール（7）、スーパー（6）、飲食店・バー（4）

> **研究上の重要な制約**：ATUSの「その他店舗・モール」には**書店が単独で分類されていない**。また、ATUSは「24時間日誌の活動時間」であり、**入店から退店までの厳密な滞在時間ではない**。まして便意は記録されていない。したがって結果から青木まりこ現象の原因や書店特有性を直接判定することはできない。

このノートブックは米国の統計データ（一次収集された公開個票を本研究で**二次分析**）をダウンロードし、集計、図示、CSV出力まで行う。日本の松田（2003）40人の研究とは標本・定義が異なるため、値を同一指標として直接比較しない。

In [ ]:
# 依存ライブラリ（Google Colab 標準環境向け）
import hashlib, io, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

YEARS = [2023, 2024, 2025]
assert YEARS == sorted(set(YEARS)), 'YEARSは重複のない昇順にしてください'
YEAR_LABEL = str(YEARS[0]) if len(YEARS) == 1 else f'{min(YEARS)}–{max(YEARS)}'
RUN_TAG = str(YEARS[0]) if len(YEARS) == 1 else f'{min(YEARS)}_{max(YEARS)}'

PROJECT_DIR = Path.cwd()
for folder in (PROJECT_DIR, *PROJECT_DIR.parents):
    if (folder / 'README.md').is_file() and (folder / 'notebooks').is_dir():
        PROJECT_DIR = folder
        break

DOWNLOAD_DIR = PROJECT_DIR / 'data' / 'raw' / 'atus'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed' / 'atus'
TABLE_DIR = PROJECT_DIR / 'results' / 'tables'
FIGURE_DIR = PROJECT_DIR / 'results' / 'figures'
NOTE_DIR = PROJECT_DIR / 'docs' / 'notes'
for directory in (DOWNLOAD_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, NOTE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

POOL_SUMMARY_PATH = TABLE_DIR / f'atus_location_summary_pooled_{RUN_TAG}.csv'
YEAR_SUMMARY_PATH = TABLE_DIR / f'atus_location_summary_by_year_{RUN_TAG}.csv'
PERSON_DAY_PATH = PROCESSED_DIR / f'atus_person_day_selected_{RUN_TAG}.csv'
BOXPLOT_PATH = FIGURE_DIR / f'atus_location_boxplot_{RUN_TAG}.png'
MEANS_PATH = FIGURE_DIR / f'atus_weighted_means_{RUN_TAG}.png'
REPORT_PATH = NOTE_DIR / f'atus_colab_report_{RUN_TAG}.md'

CATEGORIES = {
    10: 'Library',
    7: 'Other store / mall',
    6: 'Grocery store',
    4: 'Restaurant / bar',
}

print('分析対象年:', YEARS)
print('比較カテゴリ:', CATEGORIES)

## 1. 公開GitHubから公式ZIPを取得

各年の`Activity` ZIPと`Respondent` ZIPを、公開リポジトリ `data/atus/zips/` のRaw URLから認証なしで自動取得する。Colab SecretsやローカルからのZIPアップロードは不要。ファイル名・サイズ・Git blobハッシュ・ZIP整合性・内部データ名を検証し、失敗した場合は理由を表示して停止する。

In [ ]:
# GitHubで公開されているATUS ZIPを認証なしで取得する
GITHUB_REPO = 'Roki0i/aoki-mariko-research'
GITHUB_REF = 'main'
GITHUB_ZIP_DIR = 'data/atus/zips'

# GitHub登録時のファイルサイズとGit blob SHA。元データの取り違えを防ぐ
EXPECTED_ZIPS = {
    'atusact-2023.zip': (2655513, '4d93f698b0ac70bf144bf97ab172e160f01bdbff'),
    'atusact-2024.zip': (2241574, '61fb02092236951a0b4119e378e5917764b6865c'),
    'atusact-2025.zip': (1882541, 'fded17d5d605e80a30e6234cdedfae4c8eb46b95'),
    'atusresp-2023.zip': (655043, '1eae280ea2db399ec57fc7d9e8d6f240f8073210'),
    'atusresp-2024.zip': (517204, '5633fedd141f2e50d85ec86c6d4775d2c996d317'),
    'atusresp-2025.zip': (429058, '8c7b360ead8ea57065700d0cbb33e1b453f37f03'),
}

def github_failure(filename, status=None, detail=''):
    if status == 403:
        cause = 'GitHub側で公開ファイルへのアクセスが制限されています。'
    elif status == 404:
        cause = 'ファイル・ブランチが存在しないか、リポジトリが非公開です。'
    elif status is not None:
        cause = f'GitHubがHTTP {status}を返しました。'
    else:
        cause = 'GitHubへの接続またはダウンロードに失敗しました。'
    suffix = f' 詳細: {detail}' if detail else ''
    return RuntimeError(
        f'{filename} を取得できません。原因: {cause} '
        f'GitHubの公開設定と data/atus/zips/ の内容を確認してください。{suffix}'
    )

def validate_atus_zip(content, year, kind, expected_filename):
    canonical = f'atus{kind}-{year}.zip'
    if expected_filename != canonical:
        raise ValueError(f'ZIP名が不正です: {expected_filename}（期待値: {canonical}）')
    if not zipfile.is_zipfile(io.BytesIO(content)):
        raise ValueError(f'{expected_filename} はZIP形式ではありません')
    expected_member = re.compile(rf'^atus{kind}[_-]{year}\\.(?:dat|csv|txt)$', re.I)
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        bad_member = z.testzip()
        if bad_member is not None:
            raise ValueError(f'{expected_filename} のCRC検査に失敗しました: {bad_member}')
        matches = [n for n in z.namelist() if expected_member.fullmatch(Path(n).name)]
        if len(matches) != 1:
            raise ValueError(
                f'{expected_filename} 内の本体データが一意に見つかりません: {z.namelist()[:10]}'
            )
        if z.getinfo(matches[0]).file_size <= 0:
            raise ValueError(f'{expected_filename} 内の {matches[0]} が空です')
    return matches[0]

def download_github_zip(year, kind):
    filename = f'atus{kind}-{year}.zip'
    expected = EXPECTED_ZIPS.get(filename)
    if expected is None:
        raise ValueError(f'取得対象として登録されていないファイルです: {filename}')
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}/{GITHUB_ZIP_DIR}/{filename}'
    try:
        response = requests.get(url, timeout=(15, 90))
        response.raise_for_status()
    except requests.HTTPError as e:
        raise github_failure(filename, status=e.response.status_code) from e
    except requests.RequestException as e:
        raise github_failure(filename, detail=f'{type(e).__name__}: {str(e)[:160]}') from e

    content = response.content
    expected_size, expected_blob_sha = expected
    if len(content) != expected_size:
        raise github_failure(
            filename, detail=f'取得サイズ {len(content):,} bytes / 登録サイズ {expected_size:,} bytes'
        )
    blob_sha = hashlib.sha1(f'blob {len(content)}\\0'.encode() + content).hexdigest()
    if blob_sha != expected_blob_sha:
        raise github_failure(filename, detail='GitHub登録時のハッシュ値と一致しません')
    member = validate_atus_zip(content, year, kind, filename)
    print('公開GitHubから検証済みZIPを取得:', filename, f'({len(content):,} bytes; {member})')
    return content

def obtain_zip(year, kind):
    filename = f'atus{kind}-{year}.zip'
    content = download_github_zip(year, kind)
    target = DOWNLOAD_DIR / filename
    target.write_bytes(content)
    return target

def read_zip_csv(path, year, kind, columns):
    expected_member = re.compile(rf'^atus{kind}[_-]{year}\.(?:dat|csv|txt)$', re.I)
    with zipfile.ZipFile(path) as z:
        matches = [n for n in z.namelist() if expected_member.fullmatch(Path(n).name)]
        if len(matches) != 1:
            raise RuntimeError(f'{path.name} 内の本体データを一意に特定できません: {z.namelist()[:10]}')
        with z.open(matches[0]) as f:
            data = pd.read_csv(f, encoding='latin1', low_memory=False, dtype={'TUCASEID': str})
    data.columns = [str(c).strip().upper() for c in data.columns]
    missing = set(columns) - set(data.columns)
    if missing:
        raise RuntimeError(f'必要な列がありません: {missing}（実際の列例: {data.columns[:30].tolist()}）')
    return data[list(columns)].copy()

all_activity, all_respondent = [], []
for year in YEARS:
    apath = obtain_zip(year, 'act')
    rpath = obtain_zip(year, 'resp')
    a = read_zip_csv(apath, year, 'act', ['TUCASEID', 'TEWHERE', 'TUACTDUR24'])
    r = read_zip_csv(rpath, year, 'resp', ['TUCASEID', 'TUFINLWGT'])
    a['year'] = r['year'] = year
    all_activity.append(a)
    all_respondent.append(r)
    print(year, 'activity rows =', len(a), 'respondents =', len(r))

activity = pd.concat(all_activity, ignore_index=True)
respondent = pd.concat(all_respondent, ignore_index=True)
assert set(activity['year'].unique()) == set(YEARS), 'Activityの対象年が揃っていません'
assert set(respondent['year'].unique()) == set(YEARS), 'Respondentの対象年が揃っていません'
assert not respondent[['year', 'TUCASEID']].duplicated().any(), 'respondent ID重複があります'
print('全活動行:', f'{len(activity):,}', '| 調査参加者:', f'{len(respondent):,}')

## 2. 場所別の活動時間を前処理

- 活動に付いた`TEWHERE`を施設コードとして使用する。
- `TUACTDUR24`は1日の各活動の時間［分］で、**訪問1回の滞在時間とは異なる**。
- **同一人物・同一日・同一場所コードの活動時間を合計**し、「その日その分類の場所で過ごした記録時間」として分析する。
- 調査ウェイトは`TUFINLWGT`を使用。複数年結合時は年数で割り、各年の人口を足し過ぎないようにする。

In [ ]:
# 基本的なバリデーションと除外件数を保存
activity['TEWHERE'] = pd.to_numeric(activity['TEWHERE'], errors='coerce')
activity['TUACTDUR24'] = pd.to_numeric(activity['TUACTDUR24'], errors='coerce')
respondent['TUFINLWGT'] = pd.to_numeric(respondent['TUFINLWGT'], errors='coerce')

original = len(activity)
chosen = activity[activity['TEWHERE'].isin(CATEGORIES)].copy()
chosen = chosen[chosen['TUACTDUR24'].between(0.01, 1440)].copy()
print('全活動数:', original, '| 対象の場所＆有効時間を持つ活動数:', len(chosen))

chosen['category'] = chosen['TEWHERE'].astype(int).map(CATEGORIES)
chosen['duration_min'] = chosen['TUACTDUR24']

# 連続訪問を厳密に識別できないため、同日・同場所分類で合計する
person_day = (chosen.groupby(['year','TUCASEID','category'], as_index=False)
              .agg(minutes=('duration_min','sum'),
                   activity_episodes=('duration_min','size')))
person_day = person_day.merge(respondent[['year','TUCASEID','TUFINLWGT']],
                              on=['year','TUCASEID'], how='left',validate='many_to_one')
person_day = person_day[(person_day['TUFINLWGT']>0)&person_day['TUFINLWGT'].notna()]
person_day['pool_weight'] = person_day['TUFINLWGT']/len(YEARS)

print('施設カテゴリ別の有効な個人×日レコード数：', len(person_day))
print(person_day.groupby('category').agg(person_days=('TUCASEID','size'), activities=('activity_episodes','sum')))
assert (person_day['minutes']>0).all()
assert (person_day['minutes']<=1440+1e-6).all(), '1日1440分を超過しています。データ確認が必要です。'

## 3. 調査ウェイトを使った集計

各カテゴリについて、**その場所で少なくとも1つの活動を記録した人**に限定した「1日あたり記録時間」を、3年統合と年別の両方で算出する。

- n：該当カテゴリーを訪れたと記録された回答者の日誌数（無加重）
- 加重平均：BLSの最終ウェイトを使用（複数年は年数で按分）
- 中央値：加重中央値・無加重中央値の両方を表示

※複数年でも1人につき1日の日誌。年度間で同じ個人を追跡するパネルではない。標本設計を反映した厳密な信頼区間・検定は本分析には含まない。

In [ ]:
def weighted_quantile(x, w, q):
    x = np.asarray(x, float); w = np.asarray(w, float)
    valid = np.isfinite(x)&np.isfinite(w)&(w>0)
    x, w = x[valid], w[valid]
    if len(x)==0: return np.nan
    order = np.argsort(x, kind='mergesort')
    x, w = x[order], w[order]
    return float(np.interp(q*sum(w), np.cumsum(w)-0.5*w, x))

def summarize_locations(data, weight_col):
    records = []
    for code, cat in CATEGORIES.items():
        d = data[data['category'] == cat]
        if len(d) == 0:
            continue
        records.append({
            '場所': cat,
            '個人日誌数_n': len(d),
            '活動エピソード数': int(d['activity_episodes'].sum()),
            '非加重平均分': d['minutes'].mean(),
            '加重平均分': np.average(d['minutes'], weights=d[weight_col]),
            '非加重中央値分': d['minutes'].median(),
            '加重中央値分': weighted_quantile(d['minutes'], d[weight_col], 0.5),
            '加重75パーセンタイル分': weighted_quantile(d['minutes'], d[weight_col], 0.75),
            '加重90パーセンタイル分': weighted_quantile(d['minutes'], d[weight_col], 0.90),
            '最大分': d['minutes'].max(),
        })
    return pd.DataFrame(records)

summary = summarize_locations(person_day, 'pool_weight')
annual_parts = []
for year in YEARS:
    part = summarize_locations(person_day[person_day['year'] == year], 'TUFINLWGT')
    part.insert(0, '年', year)
    annual_parts.append(part)
annual_summary = pd.concat(annual_parts, ignore_index=True)
assert set(annual_summary['年']) == set(YEARS), '年別集計の対象年が揃っていません'
pd.set_option('display.float_format',lambda v:f'{v:,.2f}')
print(f'対象：{YEAR_LABEL}年、米国ATUS')
print('指標：1人1日・場所分類別の活動時間（訪問ごとの厳密滞在時間ではない）')
display(summary)
print('年別集計')
display(annual_summary)

## 4. 分布を可視化

全記録の分布を比較する。横軸は該当分類で過ごした**日誌上の合計分数**。箱ひげ図では外れ値の点表示を省略する（数値の集計には上限を適用しない）。

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
order = list(CATEGORIES.values())
labels, vals=[] , []
for cat in order:
    d = person_day[person_day.category==cat]['minutes'].to_numpy()
    if len(d):
        labels.append(cat+f' (n={len(d):,})')
        vals.append(d)
ax.boxplot(vals, vert=False, tick_labels=labels, showfliers=False)
ax.set_xlabel('Minutes per diary day in location category')
ax.set_title(f'ATUS {YEAR_LABEL}: daily activity minutes by location')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()
plt.savefig(BOXPLOT_PATH, dpi=180)
plt.show()

fig,ax=plt.subplots(figsize=(9,4.8))
plot=summary.sort_values('加重平均分')
ax.barh(plot['場所'], plot['加重平均分'])
for i, v in enumerate(plot['加重平均分']):
    ax.text(v+1,i,f'{v:.1f} min',va='center')
ax.set_xlim(0,max(plot['加重平均分'])*1.3)
ax.set_xlabel('Weighted mean minutes per diary day (among observed visitors)')
ax.set_title(f'ATUS {YEAR_LABEL} | location categories')
plt.tight_layout()
plt.savefig(MEANS_PATH, dpi=180)
plt.show()

## 5. 保存と中間報告用の解釈

- CSVは**分析済みの加工データ**。GitHubから取得したBLSの元ZIPは `data/raw/atus/` に残る。
- データを研究に使う際には、**米国の分析対象年（実行時のYEARS）、活動時間の記録、書店を単独識別できない、便意は未測定**と必ず説明する。
- 施設別の値は「その日に分類された場所で行った活動の総時間」。同日に複数訪問した場合も合算され、店舗間移動も区別できない。
- この分析は**施設滞在の背景比較**であり、青木まりこ現象そのものの実証ではない。

In [ ]:
summary.to_csv(POOL_SUMMARY_PATH, index=False, encoding='utf-8-sig')
annual_summary.to_csv(YEAR_SUMMARY_PATH, index=False, encoding='utf-8-sig')
person_day.to_csv(PERSON_DAY_PATH, index=False, encoding='utf-8-sig')

lines = [
    '# ATUS場所別活動時間・分析結果',
    f'- 対象年: {YEAR_LABEL}',
    f'- 元活動レコード数: {original:,}',
    f'- 対象の場所・有効時間を持つ活動数: {len(chosen):,}',
    f'- 場所別の個人日誌レコード数: {len(person_day):,}',
    '- 指標: 場所分類別の1日合計活動時間。1訪問あたりの滞在時間ではない。',
    '- 書店単独の識別不可、便意変数なし。原因仮説の確証には使用できない。',
    '',
    '## 3年統合の加重平均（分）',
]
lines.extend(f"- {r['場所']}: n={int(r['個人日誌数_n']):,}, 加重平均={r['加重平均分']:.2f}分, 加重中央値={r['加重中央値分']:.2f}分" for _, r in summary.iterrows())
lines.extend(['', '## 年別の加重平均（分）'])
lines.extend(f"- {int(r['年'])} {r['場所']}: n={int(r['個人日誌数_n']):,}, 加重平均={r['加重平均分']:.2f}分" for _, r in annual_summary.iterrows())
REPORT_PATH.write_text('\n'.join(lines), encoding='utf-8')
print('3年統合集計表:', POOL_SUMMARY_PATH)
print('年別集計表:', YEAR_SUMMARY_PATH)
print('個人日誌データ:', PERSON_DAY_PATH)
print('取得ZIP:', DOWNLOAD_DIR)
print('図表:', BOXPLOT_PATH, MEANS_PATH)
print('分析メモ:', REPORT_PATH)

# ダウンロード（必要な場合のみ実行）
# from google.colab import files
# files.download(str(POOL_SUMMARY_PATH))